# Q1 · Students: baselines + knowledge distillation
Requires `teacher_resnet50/best.pt` on Drive (run `colab_teacher.ipynb` first).
Expected on a T4: baselines ~1–1.5 h, KD ~1.5–2 h (all three students train together).

### Setup (Runtime → Change runtime type → **T4 GPU**)
Checkpoints and logs are written to Google Drive (`MyDrive/edge-kd-nas/`), so a disconnected
session can simply be re-run: every script resumes from its `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Private repo? use: https://<GITHUB_TOKEN>@github.com/guzorpazorp/edge_compute_assignment_4.git
REPO = 'https://github.com/guzorpazorp/edge_compute_assignment_4.git'
!test -d /content/edge-kd-nas || git clone -q $REPO /content/edge-kd-nas
%cd /content/edge-kd-nas
!git pull -q
!pip install -q optuna onnx onnxruntime onnxscript

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/edge-kd-nas'
os.environ['CKPT_DIR'] = f'{DRIVE}/checkpoints'
os.environ['RESULTS_DIR'] = f'{DRIVE}/results'
os.environ['DATA_DIR'] = '/content/data'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/download_data.py --dataset cifar100

## Baselines (hard labels only)

In [ ]:
!python scripts/train_student.py --mode baseline

## Knowledge distillation (soft targets + T + hard labels + feature KD)

In [ ]:
!python scripts/train_student.py --mode kd

In [ ]:
import pandas as pd
rows = []
for mode in ('baseline', 'kd'):
    for s in ('mbv2', 'squeezenet', 'efflite0'):
        p = f"{os.environ['RESULTS_DIR']}/logs/{mode}_{s}.csv"
        if os.path.exists(p):
            rows.append({'run': f'{mode}_{s}', 'best_top1': pd.read_csv(p).test_acc.max()})
pd.DataFrame(rows)